# 第54课 · 亲手点燃自动微分（autograd）——用一个 Value 类造出计算图（computation graph）的心脏

**目标**：实现计算图的最小单元 `Value`——存前向值与梯度，记录算子，为反向传播打地基。

> **地基课**：先给 `data / grad / _backward` 各一张「身份证」；先只做 `+` 与 `*`，再扩展。

🔗 **Aurora 连接**：`Value` 是 Aurora ML 引擎（`src/aurora/ml/`，将在 Month 2 建立）的起点；Month 2 的所有训练循环（线性层、损失函数（loss function）、优化器）都以此节点为基础构建计算图，再调用 `.backward()` 自动求导。

← **上一课**　[L53 · MFCC 图形化](../5_audio_dsp/L53_visual_mfcc.ipynb)

> 上节课学习了 **MFCC 图形化**：波形 → 声谱图 → Mel 谱 → 倒谱系数，逐层图示。  
> 本课将探讨 **Value 计算图**。

## 复习桥 · L24 链式法则（5 分钟，不跳过）

- 公式：`z = f(g(x))` → `dz/dx = (dz/dg) · (dg/dx)`
- 手算例：`z = x*y + y²`，`x=3, y=2` → `dz/dx = 2`，`dz/dy = 7`
- 本课要做的事：把这两步乘法交给 `Value.backward()` 自动完成

In [ ]:
x, y = 3.0, 2.0
z = x * y + y**2
dz_dx = y
dz_dy = x + 2 * y
assert z == 10 and dz_dx == 2 and dz_dy == 7
print("链式法则手算与 L54 断言一致")

## 📋 多元函数偏导数速查表

在高中，你只学过一元函数的导数：`y = f(x) → dy/dx`。
现在有了多元函数，比如 `z = x*y + y²`，有两个输入 x 和 y。对谁求导？

**关键思想**：对某个变量求偏导时，把其他变量看成常数。符号 `∂z/∂x` 表示"z 对 x 的偏导"。

**常见规则**（与一元导数完全类似）：

| 函数形式 | 对 x 的偏导 | 对 y 的偏导 |
|---|---|---|
| `c`（常数） | 0 | 0 |
| `x` | 1 | 0 |
| `y` | 0 | 1 |
| `x + y` | 1 | 1 |
| `x * y` | **y** | **x** |
| `x²` | **2x** | 0 |
| `y²` | 0 | **2y** |
| `x*y + y²` | **y** | **x + 2y** |

**推导 `z = x*y + y²` 的偏导** (具体演示)：
- `∂z/∂x = ∂(xy)/∂x + ∂(y²)/∂x = y + 0 = y`  ← 第一项对 x 求导时，y 当常数；第二项不含 x
- `∂z/∂y = ∂(xy)/∂y + ∂(y²)/∂y = x + 2y`  ← 第一项对 y 求导时，x 当常数；第二项是 2y

**代入数值验证** (`x=3, y=2`)：
- `∂z/∂x = y = 2` ✓
- `∂z/∂y = x + 2y = 3 + 4 = 7` ✓

## 模式对照表

| | L53 MFCC | L54 Value |
|---|---|---|
| 数据类型 | `ndarray (T, 13)` | 标量 `float` |
| 方向 | 仅前向 | 前向 + 反向 |
| 目标 | 压缩表示 | 自动求 `dL/d参数` |
| 库 | `aurora.audio` | 本课自写（L59 换 PyTorch） |

记忆点：L53 是「特征提取」，L54 是「让参数从错误中学习」。

## 本课剧情：PyTorch 的核心魔法是什么？

你写了 `loss.backward()`，PyTorch 自动算出每个参数的梯度。背后是什么？

**微积分链式法则（chain rule）**：如果 `z = f(g(x))`，则 `dz/dx = dz/dg · dg/dx`。神经网络就是一大串函数复合，层层套用链式法则，就能从输出的梯度"逆向"推导出每个参数的梯度——这叫**反向传播（backpropagation）**。

**什么是 tanh（双曲正切）？** 很多学生在这里卡住，因为高中没学过。tanh 是一个"压缩函数"，把任何数值压到 -1 到 1 之间：
- 定义：`tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))`（用欧拉数 e）
- 导数公式（不用记，课程会给）：`d(tanh(x))/dx = 1 - tanh(x)²`
- 直观理解：像 sigmoid 函数一样，把极大/极小值"挤压"到有限范围，神经网络喜欢这类激活函数

本课用 tanh 只是为了展示一个"合理的非线性函数"，具体就是：

**问题**：损失函数 `L = tanh(a*b + b²)` 有 2 个参数，手写偏导数需要链式法则分层套用：
```
∂L/∂a = (1 - tanh²(a*b+b²)) · ∂(a*b+b²)/∂a
      = (1 - tanh²(a*b+b²)) · b
∂L/∂b = (1 - tanh²(a*b+b²)) · ∂(a*b+b²)/∂b
      = (1 - tanh²(a*b+b²)) · (a + 2b)
```
这是 2 个参数的情况。神经网络有数百万参数，不可能逐一手写。

**Karpathy micrograd 的洞察**：Python 对象图 = 计算图。每次写 `c = a + b`，就隐式建了一个 `+` 节点：

```
a ──┐
    ├──[+]──► c      c.data = a.data + b.data
b ──┘         c._backward() → a.grad += 1*c.grad, b.grad += 1*c.grad
```

反向传播 = 从输出节点开始，沿有向无环图（DAG）逆向遍历，把梯度从后往前一层层传递。

本节任务：实现最小化 `Value` 类，支持 `__add__`、`__mul__`、`__pow__` 和 `backward()`。

## 🤔 为什么工程师要发明它？(Why did engineers invent this?)

- **不用它会怎样？** 每加一层网络，你就得手推一遍新的偏导公式；换个激活函数，全部重来——研究根本无法迭代。
- **它解决了什么真实问题？** 用一个 `Value` 对象把"数值 + 它是怎么算出来的"绑在一起，计算图**在你写前向代码时就自动搭好**，梯度之后能一键回传——这正是 PyTorch / TensorFlow 自动微分的最小内核。
- **后面哪里还会再用到？** L55 补算子、L56 实现 `backward()`、L57 搭 MLP、L60 换成 PyTorch autograd——全都站在这个 `Value` 类的肩膀上。

## ⚠️ 常见误解 (Common Pitfall)

> 不要把 `Value` 的 `grad` 理解成"这个数本身的值"。它存的是**损失对这个节点的偏导** `dL/d(节点)`：前向刚算完时恒为 0，只有等 `backward()` 跑完才有意义。也不要用 `=` 覆盖 `grad`——同一个节点可能被多处复用（如 `b` 在 `a*b + b**2` 里出现两次），梯度必须 `+=` 累积。

In [ ]:
# 本课纯 Python 实现，无需 numpy
# （本课不使用任何 numpy 功能——Value 类只用 Python 内建类型）

## 1. `Value` 的三个核心字段

每个 `Value` 节点携带三件事：

- `data`：前向计算的标量值，例如 `3.14`
- `grad`：该节点对最终损失 L 的偏导（partial derivative） `dL/d(self)`，**初始化为 0.0**（累积语义，不覆盖）
- `_backward`：一个**闭包（closure）**，调用时把当前节点的 `grad` 按链式法则传播给它的输入节点

`grad` 初始化为 `0.0` 而非 `None`，是因为同一个节点可能被多条路径用到（如 `b` 在 `a*b + b**2` 中出现两次），梯度需要**累加**而非覆盖。

In [ ]:
# 演示：grad 累积的必要性与具体计算
# 情景：b 被用了两次（在 a*b 和 b² 中），两条路径的梯度必须相加

# 取具体数值：a=3, b=2
a_val = 3.0
b_val = 2.0

# 前向计算
L_val = a_val * b_val + b_val ** 2  # L = 3*2 + 4 = 10

# 反向传播手算：
# 根据多元偏导数规则
# ∂L/∂b = ∂(a*b)/∂b + ∂(b²)/∂b = a + 2b = 3 + 2*2 = 7

# 但这个 7 来自哪两条路径？
# 路径1：L 通过 (a*b) 这项依赖 b → 梯度贡献 = a = 3
# 路径2：L 通过 (b²) 这项依赖 b → 梯度贡献 = 2*b = 2*2 = 4
# 总梯度 = 3 + 4 = 7 ✓

b_grad_from_path1 = a_val        # a*b 对 b 的偏导
b_grad_from_path2 = 2 * b_val    # b² 对 b 的偏导

total_b_grad = b_grad_from_path1 + b_grad_from_path2

print(f"数值演示 (a={a_val}, b={b_val})：")
print(f"  L = a*b + b² = {L_val}")
print(f"  路径1 (a*b)：∂(a*b)/∂b = a = {b_grad_from_path1}")
print(f"  路径2 (b²)：∂(b²)/∂b = 2*b = {b_grad_from_path2}")
print(f"  累加：b.grad = {b_grad_from_path1} + {b_grad_from_path2} = {total_b_grad}")
print(f"  ✓ 符合 ∂L/∂b = a + 2*b = {a_val + 2*b_val}")
print()

# 关键点：如果我们不用 += 累加，而是用 = 覆盖，会发生什么
print("如果错误地用 '=' 而不是 '+='：")
print(f"  第一次：b.grad = {b_grad_from_path1}  （来自 a*b）")
print(f"  第二次：b.grad = {b_grad_from_path2}  （来自 b²，覆盖了前面的！）")
print(f"  结果错了！应该是 {total_b_grad}，却得到 {b_grad_from_path2}")

## 2. 计算图是 DAG

每次写 `c = a + b`，都隐式创建了一张**有向无环图（DAG）**：

```
a ──┐
    ├─[+]─► c
b ──┘
```

- **节点**：每个 `Value` 对象
- **边**：算子（`+`、`*`、`**` 等）；边的方向是数据流方向（前向）
- **`_prev`**：每个节点存储它的直接输入节点集合（`{a, b}`），反向时靠这张图回溯

DAG 无环是关键——如果有环，梯度会无限循环。在前馈网络里，数据从输入到损失单向流动，自然是 DAG。

In [ ]:
# 用字典模拟 DAG 结构，演示拓扑关系
# （真实 Value 类在下方；此处仅展示图结构思路）
graph = {
    "c": {"op": "+", "inputs": ["a", "b"]},
    "d": {"op": "*", "inputs": ["c", "b"]},
}
def topo(node, graph, visited=None, order=None):
    if visited is None: visited, order = set(), []
    if node not in visited:
        visited.add(node)
        for inp in graph.get(node, {}).get("inputs", []):
            topo(inp, graph, visited, order)
        order.append(node)
    return order

print("拓扑序（前向）:", topo("d", graph))
print("反向传播顺序:", list(reversed(topo("d", graph))))

## Python 可变默认参数的"陷阱"与正确做法

很多人看到 `topo()` 的签名时会困惑：为什么不直接写 `visited=set(), order=[]`？

**危险的错误写法**：
```python
def topo(node, graph, visited=set(), order=[]):  # ❌ 危险！
    if node not in visited:
        visited.add(node)
        order.append(node)
    return order
```

问题是：Python 在**定义函数时**（不是调用时）就创建了这个默认值的**单一对象**。所有调用都共享这个对象！

```python
result1 = topo("a", {})     # visited 和 order 创建一次
result2 = topo("b", {})     # 复用同一个 visited 和 order！结果会混在一起
print(result1)              # ["a", "b"]（错了！应该只有"a"）
```

**正确的做法**：
```python
def topo(node, graph, visited=None, order=None):
    if visited is None:
        visited = set()
    if order is None:
        order = []
    # ... 后续代码
```

这样每次调用都创建新的 `set()` 和 `[]`，互不干扰。这是 Python 的标准惯用法（idiom）。

在我们的 Value 反向传播里，这很关键：每次调用 `backward()`，都需要全新的 `visited` 集合来追踪哪些节点已处理过。

## 拓扑排序与反向传播顺序

反向传播为什么要按照特定的顺序（不是随意顺序）调用 `_backward()`？

**类比**：想象工厂的生产线：
- 前向：原料 → A车间 → B车间 → C车间 → 成品
- 反向检验：必须先检查成品质量，再反向到C车间（它的产物是成品），再到B车间，最后到A车间

如果你先检查A车间，但B车间的输出还没检查，你怎么知道是A出的问题还是B放错了输入？

**数学上**：我们需要保证：当调用某个节点的 `_backward()` 时，它的输出节点（图中它的"下游"）的梯度已经全部计算完了。这样 `out.grad` 才有有效的值。

**拓扑排序的定义**：对图进行DFS（深度优先遍历），"先处理输入节点，再加入自己"，得到的序列就满足这个性质。然后反转这个序列，就是反向传播的正确顺序。

举例（图）：
```
    a ──┐
        ├──[+]──► c ──┐
    b ──┘            │
                     ├──[*]──► e
    d ────────────────┘
```

前向拓扑序（DFS后序）：a, b, c, d, e
反向传播顺序（反转）：e, d, c, b, a

这样当处理 c 的 `_backward()` 时，e 的梯度已经有了（e 依赖 c 的输出）。

## 链式法则在代码中的具体体现

在数学上，链式法则是：`dL/da = (dL/dout) · (dout/da)`。

在反向传播代码里，这变成什么样？

假设 `out = a + b`（加法），反向传播时：
- `out.grad` 已经有了值（比如是 1.0 或更复杂的数值），这就是 `dL/dout`
- 加法的局部偏导：`dout/da = 1`，`dout/db = 1`
- 根据链式法则：
  - `da_contrib = (dL/dout) * (dout/da) = out.grad * 1 = out.grad`
  - `db_contrib = (dL/dout) * (dout/db) = out.grad * 1 = out.grad`
- 所以 `a.grad += out.grad` 和 `b.grad += out.grad`

再看乘法 `out = a * b`：
- 局部偏导：`dout/da = b`，`dout/db = a`
- 链式法则：
  - `da_contrib = (dL/dout) * (dout/da) = out.grad * b = out.grad * b.data`
  - `db_contrib = (dL/dout) * (dout/db) = out.grad * a = out.grad * a.data`
- 所以 `a.grad += b.data * out.grad` 和 `b.grad += a.data * out.grad`

**关键点**：`_backward` 里那个乘以 `out.grad` 的操作，就是在执行链式法则的那一次乘法！

## 3. 为什么用 Python 类

Karpathy micrograd 的核心洞察：**Python 对象图 = 计算图**。

不需要专门的图数据库，只需重载 `__add__`、`__mul__` 等运算符——每次运算自动创建新节点并记录父子关系。Python 的垃圾回收负责内存，`_prev` 集合维护图结构，`_backward` 闭包捕获所有需要的局部变量。

对比手动求导：假设 `L = x * y + y**2`，手写 `dL/dx = y`、`dL/dy = x + 2y`。当网络有百万参数时，手写不可行——`Value` 让每个算子只需知道自己的**一阶局部偏导（local partial derivative）**，反向传播自动组合它们。

## Python 闭包（closure）速成指南

你可能在之前听过"闭包"这个概念，但具体怎么用在 `_backward` 里？

**最简单的理解**：一个函数可以定义在另一个函数内部，并且"记住"外层函数的变量，即使外层函数已经执行完了。例：

```python
def make_adder(n):
    def add_x(x):        # 内层函数
        return x + n     # 用了外层函数的参数 n
    return add_x

add_5 = make_adder(5)
print(add_5(3))          # 输出 8（3+5）
```

这里 `add_x` 就是一个闭包——它"闭包了"变量 `n`。

**在 Value 类里的应用**：

```python
def __add__(self, other):
    out = Value(...)
    
    # 这个内层函数就是闭包，它"记住"了 self, other, out
    def _backward():
        self.grad += out.grad
        other.grad += out.grad
    
    out._backward = _backward
    return out
```

当 `__add__` 执行完返回 `out` 时，`_backward` 闭包仍然"记得" `self`、`other`、`out` 这三个对象是谁，即使它们后来被用在了其他地方。等反向传播时调用 `out._backward()`，那三个对象的梯度就被正确地更新了。

**为什么不用 lambda**？试试看：

```python
# ❌ 错误尝试
_backward = lambda: (self.grad += out.grad)
# 语法错误：`+=` 是语句，不能出现在 lambda 的返回表达式里
```

所以必须定义一个真正的函数体，用语句而不是表达式。

In [ ]:
# 运算符重载的语法糖演示（伪代码对比）
print("手动求导（不可扩展）：")
print("  L = x*y + y**2")
print("  dL/dx = y")
print("  dL/dy = x + 2*y")
print()
print("Value 方案：每个算子只写一次局部偏导，链式法则自动组合")
print("  __mul__ 的 _backward: a.grad += b.data * out.grad")
print("                         b.grad += a.data * out.grad")

## 半步练习 A · 只做前向（15 分钟）

先只完成 `__init__`、`__add__`、`__mul__`，不要急着补 `__pow__` 和 `backward()`。
目标是先让前向值跑通，再去补反向传播，这样就不会一次撞上两个难点。

## 4. ✏️ 实现 `class Value`

**三个核心字段**：

| 字段 | 含义 | 初始值 |
|---|---|---|
| `data` | 标量前向值 | 构造时传入 |
| `grad` | 对最终损失的梯度 | 0.0（等待反向传播填写） |
| `_backward` | 该算子的反向函数 | `lambda: None` |

**四步实现路线**：

| 步骤 | 方法 | backward 公式 |
|---|---|---|
| 1 | `__init__` | — |
| 2 | `__add__` | `a.grad += out.grad`，`b.grad += out.grad` |
| 3 | `__mul__` | `a.grad += b.data * out.grad`，`b.grad += a.data * out.grad` |
| 4 | `backward()` | 拓扑排序（topological sort） → 逆序调用每个节点的 `_backward()` |

**验收标准**：
- `(Value(2) + Value(3)).data == 5`
- `L = a*b + b**2`，`L.backward()`，`a.grad == b.data`（对a的偏导=b）

**铺垫**：先完成上面的半步练习 A，再补 `__pow__` 和 `backward()`；前向断言先通，反向断言后通。

## Value 类的四个核心字段详解

实现 `__init__` 时，需要存储这些字段：

| 字段 | 类型 | 含义 | 初始值或来源 |
|---|---|---|---|
| `data` | `float` | 前向计算的标量值 | 构造函数参数，转成 float |
| `grad` | `float` | 对损失L的偏导 `dL/dself` | 初始为 0.0（后续反向传播时填充） |
| `_backward` | `function` | 反向传播时调用的闭包 | 初始为空函数 `lambda: None`；在 `__add__`/`__mul__` 等里重新赋值 |
| `_prev` | `set` | 这个节点的"父节点"（直接输入） | 由 `_children` 参数转换而来 |
| `_op` | `str` | 生成这个节点的算子名称 | 如 `"+"`, `"*"`, `"**"`；用于调试 |

**`_children` 参数的生命周期**：

当你在 `__add__` 里创建输出节点时：
```python
def __add__(self, other):
    out = Value(
        self.data + other.data,
        _children=(self, other),      # ← 指定这个节点的"父亲"是谁
        _op="+"                       # ← 记录它是怎么生成的
    )
    # ... 然后定义 _backward ...
    return out
```

在 `__init__` 里：
```python
def __init__(self, data, _children=(), _op=""):
    self.data = float(data)
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)    # ← 把 _children 转成 set
    self._op = _op
```

## 算子的 _backward 推导：幂函数与乘法

### 幂函数 (power function)

**局部偏导**（高中学过）：如果 `out = x^n`，则 `dout/dx = n * x^(n-1)`

举例：`y = x³` → `dy/dx = 3*x²`；在 x=2 处，`dy/dx = 3*4 = 12`

**在反向传播中**：设 `out = self^other`（where `other` 是常数，如 `2` 或 `3`），则：
```
dL/dself = (dL/dout) * (dout/dself)
         = out.grad * other * self.data^(other-1)
```

所以 `_backward` 写成：
```python
def _backward():
    self.grad += other * self.data**(other-1) * out.grad
```

### 乘法 (multiplication)

**局部偏导**：如果 `out = a * b`，则：
- `dout/da = b`
- `dout/db = a`

**在反向传播中**：
```
dL/da = (dL/dout) * (dout/da) = out.grad * b.data
dL/db = (dL/dout) * (dout/db) = out.grad * a.data
```

所以 `_backward` 写成：
```python
def _backward():
    a.grad += b.data * out.grad
    b.grad += a.data * out.grad
```

**为什么用 `b.data` 而不是 `b`？** 因为 `b` 是 Value 对象，而我们需要它的数值 `b.data` 来乘以梯度。

### 关键词汇速记

| 术语 | 含义 |
|---|---|
| `out.grad` | 损失对当前节点输出的偏导，来自下游 |
| 局部偏导（local partial） | 这个节点自己的偏导（只看这个算子，不看下游） |
| `a.grad +=` | 累加梯度（可能来自多条路径） |

## 为什么 backward() 要先设 self.grad = 1.0？

反向传播需要一个"起点"梯度。当我们调用 `loss.backward()` 时：
- 我们要求的是"损失对所有参数的偏导"
- 而损失对自己的偏导就是 1（这在数学上显然：`dL/dL = 1`）

所以我们从 `loss.grad = 1.0` 开始，然后链式法则把这个 1.0 逐层乘下去，最终在每个叶节点累积出最终的梯度。

**具体例子**（L = (a * b) + b²，a=3, b=2）：

```
前向：
  a=3.0 ──┐
          ├──[*]──► tmp1=6.0 ──┐
  b=2.0 ──┘                      │
                                  ├──[+]──► L=10.0
          b=2.0 ──┐              │
                  ├──[²]──► tmp2=4.0 ──┘

反向（逆序调用 _backward）：
  1) L.grad = 1.0  ← 初始化起点
  2) [+] 的 _backward：
     tmp1.grad += 1.0 * 1 = 1.0
     tmp2.grad += 1.0 * 1 = 1.0
  3) [*] 的 _backward（计算tmp1=a*b）：
     a.grad += b.data * tmp1.grad = 2.0 * 1.0 = 2.0
     b.grad += a.data * tmp1.grad = 3.0 * 1.0 = 3.0
  4) [²] 的 _backward（计算tmp2=b**2）：
     b.grad += 2 * b.data * tmp2.grad = 2 * 2.0 * 1.0 = 4.0
  
  最终：a.grad = 2.0，b.grad = 3.0 + 4.0 = 7.0
```

看出来了吗？`L.grad = 1.0` 就像是第一次乘法中的"乘数"，然后沿着计算图反向传播，每一层都乘以那一层的局部偏导。

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        # ✏️ TODO: 存 data（转 float）、grad=0.0、_backward=lambda:None
        #          _prev=set(_children)、_op=_op
        raise NotImplementedError("请实现 __init__：设置 self.data, self.grad, self._backward, self._prev, self._op")

    def __repr__(self):
        # 使用 getattr 防止在桩状态（__init__ 未实现时）引发 AttributeError
        return f"Value(data={getattr(self, 'data', '?')}, grad={getattr(self, 'grad', '?')})"

    def __add__(self, other):
        # ✏️ TODO: other 若不是 Value 则包装；创建 out；定义 _backward（加法偏导均为1）
        raise NotImplementedError("请实现 __add__：包装 other，创建 out，设置 _backward")

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        # ✏️ TODO: 乘积法则 _backward
        raise NotImplementedError("请实现 __mul__：包装 other，创建 out，用乘积法则设置 _backward")

    def __rmul__(self, other): return self * other

    def __pow__(self, other):
        # ✏️ TODO: 幂函数偏导：other * self.data**(other-1)
        assert isinstance(other, (int, float)), "指数必须是标量"
        raise NotImplementedError("请实现 __pow__：创建 out，设置 _backward（偏导 = other * self.data**(other-1)）")

    def __neg__(self):  return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**-1

    def backward(self):
        # ✏️ TODO: 拓扑排序整图，逆序调用 _backward；先设 self.grad=1.0
        # 提示：参考 Cell 6 的 topo() 函数，用 DFS + visited 集合收集后序节点，再反转
        raise NotImplementedError("请实现 backward：先 self.grad=1.0，拓扑排序后逆序调用各节点 _backward")

In [ ]:
# 前向半步：只要 __init__ / __add__ / __mul__ 完成，这段就应先通过
a = Value(2.0)
b = Value(3.0)
c = a + b
assert c.data == 5.0, f"期望 5.0，得到 {c.data}"
print("✅ a + b 前向正确：", c)

d = a * b
assert d.data == 6.0, f"期望 6.0，得到 {d.data}"
print("✅ a * b 前向正确：", d)

# 完整 backward：如果 __pow__ / backward 还没补完，这段会友好提示
try:
    a2 = Value(2.0)
    b2 = Value(3.0)
    out = a2 * b2
    out.grad = 1.0
    out._backward()
    assert a2.grad == 3.0, f"期望 a.grad=3.0，得到 {a2.grad}"
    assert b2.grad == 2.0, f"期望 b.grad=2.0，得到 {b2.grad}"
    print("✅ __mul__ _backward 正确：a.grad=", a2.grad, " b.grad=", b2.grad)

    a3 = Value(4.0)
    b3 = Value(2.0)
    L = a3 * b3 + b3 ** 2   # L = 4*2 + 4 = 12；dL/da=2, dL/db=4+4=8
    L.backward()
    assert a3.grad == 2.0, f"期望 dL/da=2.0，得到 {a3.grad}"
    assert b3.grad == 8.0, f"期望 dL/db=8.0，得到 {b3.grad}"
    print("✅ L=(a*b + b**2) backward() 正确：a.grad=", a3.grad, " b.grad=", b3.grad)
except (NotImplementedError, TypeError):
    print("⬜ backward / __pow__ 还未完成，先把前向半步补完即可继续。")

## 5. 参数实验：手算 vs 自动微分

构建 `L = (a * b) + b**2`，取 `a=3.0, b=2.0`。

**手算**：
- `L = 3*2 + 2**2 = 10`
- `dL/da = b = 2`
- `dL/db = a + 2*b = 3 + 4 = 7`

**预期现象**：调用 `L.backward()` 后，`a.grad==2.0`，`b.grad==7.0`。

改变 `a` 和 `b` 的值，观察梯度如何随输入线性/非线性变化。

## 完整推导：L = (a*b) + b² 的梯度

这是最重要的一个例子。我们要完整推导出梯度公式。

**第一步：确定函数形式**
```
L = (a*b) + b²
```

**第二步：分别对 a 和 b 求偏导**（使用多元偏导数规则）

对 a 求偏导：
```
∂L/∂a = ∂(a*b)/∂a + ∂(b²)/∂a
      = b + 0                   （第一项：b是常数时a的系数；第二项不含a）
      = b
```

对 b 求偏导（这里需要链式法则组合两条路径）：
```
∂L/∂b = ∂(a*b)/∂b + ∂(b²)/∂b
      = a + 2*b                 （第一项：a是常数时b的系数；第二项：2*b）
```

**第三步：代入数值验证** (a=3, b=2)
```
L = 3*2 + 2² = 6 + 4 = 10 ✓
∂L/∂a = b = 2 ✓
∂L/∂b = a + 2*b = 3 + 4 = 7 ✓
```

**第四步：理解梯度累积**

b 的梯度 7 来自哪里？
- 路径1：L 中的 (a*b) 这一项对 b 的贡献 = a = 3
- 路径2：L 中的 (b²) 这一项对 b 的贡献 = 2*b = 4
- 总计：3 + 4 = 7

在反向传播代码里，这就是为什么我们要用 `+=` 累加而不是 `=` 覆盖！

In [ ]:
# 参数实验：改变 a, b 的值，验证梯度公式
for a_val, b_val in [(3.0, 2.0), (1.0, 5.0), (-2.0, 3.0)]:
    a = Value(a_val); b = Value(b_val)
    L = a * b + b ** 2
    L.backward()
    # 手算期望
    expected_da = b_val
    expected_db = a_val + 2 * b_val
    print(f"a={a_val}, b={b_val} → L={L.data:.1f} | "
          f"a.grad={a.grad:.1f}（期望{expected_da:.1f}） "
          f"b.grad={b.grad:.1f}（期望{expected_db:.1f}）")
    assert abs(a.grad - expected_da) < 1e-9
    assert abs(b.grad - expected_db) < 1e-9
print("✅ 全部参数组合验证通过")

## 🎯 未来的回报 (Future Payoff)

今天你亲手造的 `Value` 类，会在 **L55 / L56 / L60** 再次出现——L55 给它补满算子，L56 让它学会自动 `backward()`，到 L60 你会在 PyTorch 里一眼认出同一套 `.grad` / `.backward()` 设计。从零写过一遍，框架对你就不再是黑箱。

## 本课收束

`Value` 类通过 `__add__`、`__mul__`、`__pow__` 三个运算符重载构建计算图，每个算子在 `_backward` 闭包里写入自己的局部偏导，`backward()` 按拓扑逆序触发所有 `_backward`，最终在每个叶节点的 `.grad` 里得到 `dL/d(leaf)`。这套机制输出的是**标量梯度**，对应 Aurora ML 引擎的核心概念（`src/aurora/ml/` 为计划中模块，尚未建立；当前自动微分（automatic differentiation，autograd）实现直接在 notebook 中演示）。下一节（**L55**）将补全 `__pow__`、`tanh`、`relu`、`exp` 算子，把计算图的算子库配齐；`Neuron → Layer → MLP` 的搭建在 L57，真正的训练循环在 L58。

## ✏️ 闭卷推导检查格 — 计算图反向传播

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：给定表达式 $z = x \cdot y + y^2$，其中 $x = 3, y = 2$：

1. 画出前向计算图（标注所有中间节点的值）
2. 手写出链式展开，求 $\partial z / \partial x$ 和 $\partial z / \partial y$
3. 验证你的答案：$\partial z / \partial x = ?$，$\partial z / \partial y = ?$

（在此处写推导...）

In [ ]:
# 验证：优先使用本 notebook 的 Value；若还没实现完，则回退到数值差分做对照（纯 Python，无需额外依赖）
try:
    x = Value(3.0)
    y = Value(2.0)
    z = x * y + y ** 2
    z.backward()
    dz_dx = x.grad
    dz_dy = y.grad
except (NotImplementedError, TypeError):
    # 中心差分：f(x+h) 与 f(x-h) 的斜率逼近导数（对二次式来说几乎精确）
    f = lambda x, y: x * y + y ** 2
    h = 1e-6
    dz_dx = (f(3.0 + h, 2.0) - f(3.0 - h, 2.0)) / (2 * h)
    dz_dy = (f(3.0, 2.0 + h) - f(3.0, 2.0 - h)) / (2 * h)

# z = xy + y^2 → dz/dx = y = 2, dz/dy = x + 2y = 3 + 4 = 7
assert abs(dz_dx - 2.0) < 1e-6, f"dz/dx = {dz_dx}，期望 2"
assert abs(dz_dy - 7.0) < 1e-6, f"dz/dy = {dz_dy}，期望 7"
print(f"✅ dz/dx = {dz_dx} ✓   dz/dy = {dz_dy} ✓")

In [ ]:
# ✏️ 本课自评
l54_review = {
    "chain_rule_understood":   None,  # 理解链式法则：dz/dx=dz/dg·dg/dx？True/False
    "value_class_impl":        None,  # Value.__add__/__mul__/__pow__/backward 全实现？True/False
    "backprop_dag_order":      None,  # 理解反向传播=拓扑逆序调用_backward？True/False
    "grad_accumulation":       None,  # 理解同一节点被多次使用时 grad 需要累加？True/False
    "whiteboard_passed":       None,  # 白板挑战计算图反向传播推导通过？True/False
}

unfilled = [k for k, v in l54_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l54_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L54 全部通关！进入 L55：Value 算子补全')

---

→ **下一课**　[L55 · Value 算子补全](L55_forward_pass.ipynb)

> 下节课将学习 **Value 算子补全**：pow、relu、tanh、exp 节点实现，计算图完整展开。